# Импорты

In [34]:
import pandas
import numpy

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

# Загрузка

In [35]:
df = pandas.read_csv("Wrongful Convictions in the United States.csv")
df

,Unnamed: 0,Date of the Crime,Defendants,Type of the Crime,Location of the Crime,Punishment for a crime,Legally Exonerated,Legally Exonerated Encoded
0,0,1805,Dominic Daley and James Halligan,murder,Massachusetts,death,yes,1
1,1,1843,John Gordon,murder,Rhode Island,death,yes,1
2,2,1855,Chief Leschi,murder,Washington,death,no,0
3,3,1863,Chipita Rodriguez,murder,Texas,death,yes,1
4,4,1872,William Jackson Marion,murder,Nebraska,death,yes,1
...,...,...,...,...,...,...,...,...
173,173,2016,Tazell Cash,"robbery, other violent felony",Michigan,20 years,yes,1
174,174,2017,Alex Heineman,sexual assault,Wisconsin,NaN,yes,1
175,175,2017,Michael Hickingbottom,assault,Indiana,6 years,yes,1
176,176,2017,Joshua Horner,child sex abuse,Oregon,50 years,yes,1


# Предобработка

## Удаление лишних столбцов

In [36]:
df.drop(columns=["Legally Exonerated", "Unnamed: 0"], inplace=True)
df

,Date of the Crime,Defendants,Type of the Crime,Location of the Crime,Punishment for a crime,Legally Exonerated Encoded
0,1805,Dominic Daley and James Halligan,murder,Massachusetts,death,1
1,1843,John Gordon,murder,Rhode Island,death,1
2,1855,Chief Leschi,murder,Washington,death,0
3,1863,Chipita Rodriguez,murder,Texas,death,1
4,1872,William Jackson Marion,murder,Nebraska,death,1
...,...,...,...,...,...,...
173,2016,Tazell Cash,"robbery, other violent felony",Michigan,20 years,1
174,2017,Alex Heineman,sexual assault,Wisconsin,NaN,1
175,2017,Michael Hickingbottom,assault,Indiana,6 years,1
176,2017,Joshua Horner,child sex abuse,Oregon,50 years,1


## Обработка пропусков

In [37]:
df.isna().sum()

Date of the Crime             0
Defendants                    0
Type of the Crime             0
Location of the Crime         0
Punishment for a crime        9
Legally Exonerated Encoded    0
dtype: int64

In [38]:
df[df["Punishment for a crime"].isna()]

,Date of the Crime,Defendants,Type of the Crime,Location of the Crime,Punishment for a crime,Legally Exonerated Encoded
7,1894,George Washington Davis,sabotage of Locomotive 213,Nebraska,NaN,1
9,1900,Caleb Powers,murder,Kentucky,NaN,1
50,1980,Steve Titus,rape,Washington,NaN,1
52,1980,Kern County child abuse cases,satanic ritual sex abuse,California,NaN,1
71,1984,Employees of Fells Acres Day Care,satanic ritual sex abuse,Massachusetts,NaN,1
79,1985,Beatrice Six,murder,Nebraska,NaN,1
100,1989,"Central Park Five: Yusef Salaam, Antron McCray...","assault, rape",New York,NaN,1
110,1991,Dixmoor 5,murder,Illinois,NaN,1
174,2017,Alex Heineman,sexual assault,Wisconsin,NaN,1


В качестве замены пропуска в "Punishment for a crime" будет самое популярное значение "Type of the Crime" для соответствующего значения "Punishment for a crime"

In [39]:
for feat in df.loc[df["Punishment for a crime"].isna(), "Type of the Crime"].unique():
    most_popular = df.loc[df["Type of the Crime"] == feat, "Punishment for a crime"].mode()

    if not most_popular.empty:
        most_popular = most_popular[0]
        
        df.loc[
            df["Punishment for a crime"].isna() & (df["Type of the Crime"] == feat),
            "Punishment for a crime"
        ] = most_popular

        print(f"{most_popular} -> {feat}")

df.isna().sum()

death -> murder
50 years -> rape
7 years -> satanic ritual sex abuse
life in prison -> assault, rape
12.5 years -> sexual assault


Date of the Crime             0
Defendants                    0
Type of the Crime             0
Location of the Crime         0
Punishment for a crime        1
Legally Exonerated Encoded    0
dtype: int64

In [40]:
df[df["Punishment for a crime"].isna()]

,Date of the Crime,Defendants,Type of the Crime,Location of the Crime,Punishment for a crime,Legally Exonerated Encoded
7,1894,George Washington Davis,sabotage of Locomotive 213,Nebraska,NaN,1


In [41]:
df["Type of the Crime"].unique()

array(['murder', 'haymarket affair', 'sabotage of Locomotive 213', 'rape',
       'preparedness Day Bombing', 'breaking and entering, petty theft',
       'sexual assault', 'robbery, rape', 'murder, robbery',
       'satanic ritual sex abuse', 'rape,  murder', 'assault, rape',
       'murder, robbery, burglary', 'rape, sodomy, robbery',
       'rape, robbery', 'child sexual abuse', 'sexual abuse of children',
       'murder, robbery, illegal use of a weapon', 'murder, carjacking',
       'sexual assault, sexual abuse, kidnapping, robbery',
       'rape, kidnapping',
       'sexual assault, robbery, other violent felony',
       'drive-by shooting', 'rape, murder',
       'criminal sexual conduct, armed robbery', 'filing a false report',
       'murder, rape', 'sexual assault, kidnapping', 'child abuse',
       'manslaughter', 'manslaughter of infant', 'child sex abuse',
       'weapon possession or sale',
       'robbery, assault, illegal use of a weapon', 'robbery',
       'murder, as

Было принято решение, в качестве "Punishment for a crime" для "Type of the Crime" = "sabotage of Locomotive 213" подставить значение "Punishment for a crime" "Type of the Crime" = "preparedness Day Bombing", так как предположительно оба преступления относятся к преступлению с применением взрывчатки.

In [42]:
df.fillna(df.loc[df["Type of the Crime"] == "preparedness Day Bombing", "Punishment for a crime"].unique()[0], inplace=True)
df.isna().sum()

Date of the Crime             0
Defendants                    0
Type of the Crime             0
Location of the Crime         0
Punishment for a crime        0
Legally Exonerated Encoded    0
dtype: int64

## Обработка дубликатов

In [43]:
df.duplicated().sum()

np.int64(0)

## Обработка категориальных признаков

In [44]:
df_without_processing = df.copy()
df_without_processing

,Date of the Crime,Defendants,Type of the Crime,Location of the Crime,Punishment for a crime,Legally Exonerated Encoded
0,1805,Dominic Daley and James Halligan,murder,Massachusetts,death,1
1,1843,John Gordon,murder,Rhode Island,death,1
2,1855,Chief Leschi,murder,Washington,death,0
3,1863,Chipita Rodriguez,murder,Texas,death,1
4,1872,William Jackson Marion,murder,Nebraska,death,1
...,...,...,...,...,...,...
173,2016,Tazell Cash,"robbery, other violent felony",Michigan,20 years,1
174,2017,Alex Heineman,sexual assault,Wisconsin,12.5 years,1
175,2017,Michael Hickingbottom,assault,Indiana,6 years,1
176,2017,Joshua Horner,child sex abuse,Oregon,50 years,1


### Обработка Defendants

Обработаем имена заключённых, представив их в виде числа заключённых, участвовавших в одном преступлении

In [45]:
df["Defendants"].unique()

array(['Dominic Daley\xa0and\xa0James Halligan', 'John Gordon',
       'Chief Leschi', 'Chipita Rodriguez', 'William Jackson Marion',
       'Oscar Neebe,\xa0August Spies, and\xa0Albert Parsons',
       'Charles Hudspeth', 'George Washington Davis', 'Jack Davis',
       'Caleb Powers', 'Ed Johnson', 'Bill Wilson',
       'Thomas Griffin and Meeks Griffin', 'Leo Frank', 'Thomas Mooney',
       'Nicola Sacco and Bartolomeo Vanzet', 'Scottsboro Boys',
       'Joseph Majczek and Theodore Marcinkiewicz', 'Joe Arridy',
       'George Stinney', 'Jack McCullough', 'Clarence Earl Gideon',
       'Muhammad Aziz and Khalil Islam', 'Rubin Carter',
       'James Joseph Richardson', 'Richard Phillips', 'Anthony Mazza',
       'Wilbert Jones', 'Gregory Bright', 'Delbert Tibbs',
       'Michael Lloyd Self', 'David Bryant', 'Ledura Watkins',
       'Ricky Jackson, Ronnie Bridgeman, and Wiley Bridgeman',
       'Clifford Williams, Jr', 'Charles Ray Finch', 'Lewis Fogle',
       'Randall Dale Adams', 'De

In [46]:
for defendant in df["Defendants"].unique():
    humans = (
        defendant
        .replace("\xa0", " ")
        .replace(" and ", ", ")
        .replace(",,", ",")
        .replace(", Jr", "")
        .split(", ")
    )

    df.loc[df["Defendants"] == defendant, "Defendants"] = len(humans)

df["Defendants"] = df["Defendants"].astype(int)
df_without_processing["Defendants"] = df["Defendants"]
df

,Date of the Crime,Defendants,Type of the Crime,Location of the Crime,Punishment for a crime,Legally Exonerated Encoded
0,1805,2,murder,Massachusetts,death,1
1,1843,1,murder,Rhode Island,death,1
2,1855,1,murder,Washington,death,0
3,1863,1,murder,Texas,death,1
4,1872,1,murder,Nebraska,death,1
...,...,...,...,...,...,...
173,2016,1,"robbery, other violent felony",Michigan,20 years,1
174,2017,1,sexual assault,Wisconsin,12.5 years,1
175,2017,1,assault,Indiana,6 years,1
176,2017,1,child sex abuse,Oregon,50 years,1


### Обработка Type of the Crime

In [47]:
df["Type of the Crime"].unique()

array(['murder', 'haymarket affair', 'sabotage of Locomotive 213', 'rape',
       'preparedness Day Bombing', 'breaking and entering, petty theft',
       'sexual assault', 'robbery, rape', 'murder, robbery',
       'satanic ritual sex abuse', 'rape,  murder', 'assault, rape',
       'murder, robbery, burglary', 'rape, sodomy, robbery',
       'rape, robbery', 'child sexual abuse', 'sexual abuse of children',
       'murder, robbery, illegal use of a weapon', 'murder, carjacking',
       'sexual assault, sexual abuse, kidnapping, robbery',
       'rape, kidnapping',
       'sexual assault, robbery, other violent felony',
       'drive-by shooting', 'rape, murder',
       'criminal sexual conduct, armed robbery', 'filing a false report',
       'murder, rape', 'sexual assault, kidnapping', 'child abuse',
       'manslaughter', 'manslaughter of infant', 'child sex abuse',
       'weapon possession or sale',
       'robbery, assault, illegal use of a weapon', 'robbery',
       'murder, as

Разделим преступления на группы по общему характеру:
| Характер | Тип преступления |
| -------- | ---------------- |
| assault | murder |
| assault | illegal use of a weapon |
| assault | kidnapping |
| assault | drive-by shooting |
| assault | other violent felony |
| assault | child abuse |
| assault | manslaughter |
| assault | manslaughter of infant |
| assault | felony assault of a child |
| assault | attempted murder |
| assault | assault |
| assault | arson |
| conspiracy | haymarket affair |
| conspiracy | sabotage of Locomotive 213 |
| conspiracy | preparedness Day Bombing |
| conspiracy | filing a false report |
| conspiracy | weapon possession or sale |
| conspiracy | conspiracy |
| conspiracy | gun possession or sale and conspiracy |
| sexual assault | rape |
| sexual assault | sexual assault |
| sexual assault | satanic ritual sex abuse |
| sexual assault | sodomy |
| sexual assault | child sexual abuse |
| sexual assault | sexual abuse of children |
| sexual assault | sexual abuse |
| sexual assault | criminal sexual conduct |
| sexual assault | child sex abuse |
| theft | breaking and entering |
| theft | petty theft |
| theft | burglary |
| theft | carjacking |
| theft | fraud |
| theft | theft |
| robbery | robbery |
| robbery | armed robbery |

In [48]:
CRIMES = {
    "murder": "assault",
    "illegal use of a weapon": "assault",
    "kidnapping": "assault",
    "drive-by shooting": "assault",
    "other violent felony": "assault",
    "child abuse": "assault",
    "manslaughter": "assault",
    "manslaughter of infant": "assault",
    "felony assault of a child": "assault",
    "attempted murder": "assault",
    "assault": "assault",
    "arson": "assault",

    "haymarket affair": "conspiracy",
    "sabotage of Locomotive 213": "conspiracy",
    "preparedness Day Bombing": "conspiracy",
    "filing a false report": "conspiracy",
    "weapon possession or sale": "conspiracy",
    "conspiracy": "conspiracy",
    "gun possession or sale and conspiracy": "conspiracy",

    "rape": "sexual assault",
    "sexual assault": "sexual assault",
    "satanic ritual sex abuse": "sexual assault",
    "sodomy": "sexual assault",
    "child sexual abuse": "sexual assault",
    "sexual abuse of children": "sexual assault",
    "sexual abuse": "sexual assault",
    "criminal sexual conduct": "sexual assault",
    "child sex abuse": "sexual assault",

    "breaking and entering": "theft",
    "petty theft": "theft",
    "burglary": "theft",
    "carjacking": "theft",
    "fraud": "theft",
    "theft": "theft",

    "robbery": "robbery",
    "armed robbery": "robbery",
}

df["Type of the Crime"] = (
    df["Type of the Crime"]
    .str.split(",")
    .apply(lambda x: list({CRIMES[key.strip()] for key in x}))
)
df

,Date of the Crime,Defendants,Type of the Crime,Location of the Crime,Punishment for a crime,Legally Exonerated Encoded
0,1805,2,[assault],Massachusetts,death,1
1,1843,1,[assault],Rhode Island,death,1
2,1855,1,[assault],Washington,death,0
3,1863,1,[assault],Texas,death,1
4,1872,1,[assault],Nebraska,death,1
...,...,...,...,...,...,...
173,2016,1,"[assault, robbery]",Michigan,20 years,1
174,2017,1,[sexual assault],Wisconsin,12.5 years,1
175,2017,1,[assault],Indiana,6 years,1
176,2017,1,[sexual assault],Oregon,50 years,1


Теперь переведём в численный тип через one-hot encoding

In [49]:
df = df.join(
    df["Type of the Crime"]
    .explode()
    .pipe(pandas.get_dummies)
    .groupby(level=0)
    .max()
    .astype(int)
)
df

,Date of the Crime,Defendants,Type of the Crime,Location of the Crime,Punishment for a crime,Legally Exonerated Encoded,assault,conspiracy,robbery,sexual assault,theft
0,1805,2,[assault],Massachusetts,death,1,1,0,0,0,0
1,1843,1,[assault],Rhode Island,death,1,1,0,0,0,0
2,1855,1,[assault],Washington,death,0,1,0,0,0,0
3,1863,1,[assault],Texas,death,1,1,0,0,0,0
4,1872,1,[assault],Nebraska,death,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
173,2016,1,"[assault, robbery]",Michigan,20 years,1,1,0,1,0,0
174,2017,1,[sexual assault],Wisconsin,12.5 years,1,0,0,0,1,0
175,2017,1,[assault],Indiana,6 years,1,1,0,0,0,0
176,2017,1,[sexual assault],Oregon,50 years,1,0,0,0,1,0


In [50]:
df_without_processing["Type of the Crime"] = df["Type of the Crime"]
df = df.drop(columns=["Type of the Crime"])
df

,Date of the Crime,Defendants,Location of the Crime,Punishment for a crime,Legally Exonerated Encoded,assault,conspiracy,robbery,sexual assault,theft
0,1805,2,Massachusetts,death,1,1,0,0,0,0
1,1843,1,Rhode Island,death,1,1,0,0,0,0
2,1855,1,Washington,death,0,1,0,0,0,0
3,1863,1,Texas,death,1,1,0,0,0,0
4,1872,1,Nebraska,death,1,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
173,2016,1,Michigan,20 years,1,1,0,1,0,0
174,2017,1,Wisconsin,12.5 years,1,0,0,0,1,0
175,2017,1,Indiana,6 years,1,1,0,0,0,0
176,2017,1,Oregon,50 years,1,0,0,0,1,0


### Обработка Location of the Crime

In [51]:
df["Location of the Crime"].unique()

array(['Massachusetts', 'Rhode Island', 'Washington', 'Texas', 'Nebraska',
       'Illinois', 'Arkansas', 'Idaho', 'Kentucky', 'Tennessee',
       'Alabama', 'South Carolina', 'Georgia', 'California', 'Colorado',
       'Florida', 'New York', 'New Jersey', 'Michigan', 'Louisiana',
       'Ohio', 'North Carolina', 'Pennsylvania', 'Missouri', 'Nevada',
       'Virginia', 'Oklahoma', 'Maryland', 'Wisconsin', 'Oregon',
       'West Virginia', 'Connecticut', 'Mississippi', 'Arizona', 'Iowa',
       'Indiana', 'Montana', 'Kansas'], dtype=object)

Применяем к данным one hot encoding

In [52]:
dummies = pandas.get_dummies(
    df["Location of the Crime"],
    prefix="Location of the Crime",
    dtype=int
)

df = df.join(dummies)
df

,Date of the Crime,Defendants,Location of the Crime,Punishment for a crime,Legally Exonerated Encoded,assault,conspiracy,robbery,sexual assault,theft,...,Location of the Crime_Oregon,Location of the Crime_Pennsylvania,Location of the Crime_Rhode Island,Location of the Crime_South Carolina,Location of the Crime_Tennessee,Location of the Crime_Texas,Location of the Crime_Virginia,Location of the Crime_Washington,Location of the Crime_West Virginia,Location of the Crime_Wisconsin
0,1805,2,Massachusetts,death,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1843,1,Rhode Island,death,1,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2,1855,1,Washington,death,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,1863,1,Texas,death,1,1,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,1872,1,Nebraska,death,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,2016,1,Michigan,20 years,1,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
174,2017,1,Wisconsin,12.5 years,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,1
175,2017,1,Indiana,6 years,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
176,2017,1,Oregon,50 years,1,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,0


In [53]:
df_without_processing["Location of the Crime"] = df["Location of the Crime"]
df = df.drop(columns=["Location of the Crime"])
df

,Date of the Crime,Defendants,Punishment for a crime,Legally Exonerated Encoded,assault,conspiracy,robbery,sexual assault,theft,Location of the Crime_Alabama,...,Location of the Crime_Oregon,Location of the Crime_Pennsylvania,Location of the Crime_Rhode Island,Location of the Crime_South Carolina,Location of the Crime_Tennessee,Location of the Crime_Texas,Location of the Crime_Virginia,Location of the Crime_Washington,Location of the Crime_West Virginia,Location of the Crime_Wisconsin
0,1805,2,death,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1843,1,death,1,1,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2,1855,1,death,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,1863,1,death,1,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,1872,1,death,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,2016,1,20 years,1,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
174,2017,1,12.5 years,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
175,2017,1,6 years,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
176,2017,1,50 years,1,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0


### Обработка Punishment for a crime

In [54]:
df["Punishment for a crime"].unique()

array(['death', '15 years', 'life in prison', '5 years',
       'life without parole', '25 years', '20 years', '50 years',
       'death for Jimerson and Williams, 75 years for Adams, and life for Rainge',
       '75 years', '7 years', '30 years', '40 years', '32 years',
       '63 years', '45 years', '33 years', '35 years', '11-35 years',
       '84 years', '49 years', '10 years', '57 years', '60 years',
       'community service', '8 years', '65 years', '55 years', '17 years',
       'probation, lifetime sex offender registration', '6 years',
       '22 years', 'present', '21 years', '0.5 years', '4 years',
       '1.2 year', '16 years', '26 years', '12.5 years', 'not sentenced',
       '9 months'], dtype=object)

Для начала обработаем выбросы

In [55]:
df["Punishment for a crime"] = df["Punishment for a crime"].replace({
    "11-35 years": "23 years",
    "death for Jimerson and Williams, 75 years for Adams, and life for Rainge": "death, 75 years, life in prison",
    "1.2 year": "1.2 years",
    "not sentenced": "0 years",
    "9 months": "0.75 years"
})
df["Punishment for a crime"].unique()

array(['death', '15 years', 'life in prison', '5 years',
       'life without parole', '25 years', '20 years', '50 years',
       'death, 75 years, life in prison', '75 years', '7 years',
       '30 years', '40 years', '32 years', '63 years', '45 years',
       '33 years', '35 years', '23 years', '84 years', '49 years',
       '10 years', '57 years', '60 years', 'community service', '8 years',
       '65 years', '55 years', '17 years',
       'probation, lifetime sex offender registration', '6 years',
       '22 years', 'present', '21 years', '0.5 years', '4 years',
       '1.2 years', '16 years', '26 years', '12.5 years', '0 years',
       '0.75 years'], dtype=object)

In [56]:
df.loc[df["Punishment for a crime"] == "present", "Punishment for a crime"] = f"{2026 - df.loc[df["Punishment for a crime"] == "present", "Date of the Crime"].unique()[0]} years"
df["Punishment for a crime"].unique()

array(['death', '15 years', 'life in prison', '5 years',
       'life without parole', '25 years', '20 years', '50 years',
       'death, 75 years, life in prison', '75 years', '7 years',
       '30 years', '40 years', '32 years', '63 years', '45 years',
       '33 years', '35 years', '23 years', '84 years', '49 years',
       '10 years', '57 years', '60 years', 'community service', '8 years',
       '65 years', '55 years', '17 years',
       'probation, lifetime sex offender registration', '6 years',
       '22 years', '21 years', '0.5 years', '4 years', '1.2 years',
       '16 years', '26 years', '12.5 years', '0 years', '0.75 years'],
      dtype=object)

Для перевода в численный вид, было принято решение применить смесь one-hot encoding и label encoding.</br>
Видов наказаний будет 4 группы: death, life sentence, alternative punishment, imprisonment.</br>
death будет принимать 1, если значение включает "death", 0 $-$ в ином случае.</br>
life sentence будет принимать значение 1, если включает "life in prison", 2, если включает "life without parole", 0 $-$ в ином случае.</br>
alternative punishment будет принимать значение 1, если включает "community service", 2, если включает "probation, lifetime sex offender registration", 0 $-$ в ином случае.</br>
imprisonment будет принимать значение срока в тюрьме, если это не пожизненное.

In [57]:
df["death"] = (df["Punishment for a crime"].str.contains("death")).astype(int)
df

,Date of the Crime,Defendants,Punishment for a crime,Legally Exonerated Encoded,assault,conspiracy,robbery,sexual assault,theft,Location of the Crime_Alabama,...,Location of the Crime_Pennsylvania,Location of the Crime_Rhode Island,Location of the Crime_South Carolina,Location of the Crime_Tennessee,Location of the Crime_Texas,Location of the Crime_Virginia,Location of the Crime_Washington,Location of the Crime_West Virginia,Location of the Crime_Wisconsin,death
0,1805,2,death,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,1843,1,death,1,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,1
2,1855,1,death,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,1
3,1863,1,death,1,1,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,1
4,1872,1,death,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,2016,1,20 years,1,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
174,2017,1,12.5 years,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0
175,2017,1,6 years,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
176,2017,1,50 years,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [58]:
df["life sentence"] = numpy.select(
    [
        df["Punishment for a crime"].str.contains("life without parole", na=False),
        df["Punishment for a crime"].str.contains("life in prison", na=False),
    ],
    [
        2,  # life without parole
        1,  # life in prison
    ],
    default=0
)
df

,Date of the Crime,Defendants,Punishment for a crime,Legally Exonerated Encoded,assault,conspiracy,robbery,sexual assault,theft,Location of the Crime_Alabama,...,Location of the Crime_Rhode Island,Location of the Crime_South Carolina,Location of the Crime_Tennessee,Location of the Crime_Texas,Location of the Crime_Virginia,Location of the Crime_Washington,Location of the Crime_West Virginia,Location of the Crime_Wisconsin,death,life sentence
0,1805,2,death,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,1843,1,death,1,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,1,0
2,1855,1,death,0,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,1,0
3,1863,1,death,1,1,0,0,0,0,0,...,0,0,0,1,0,0,0,0,1,0
4,1872,1,death,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,2016,1,20 years,1,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
174,2017,1,12.5 years,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,0
175,2017,1,6 years,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
176,2017,1,50 years,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [59]:
df["alternative punishment"] = numpy.select(
    [
        df["Punishment for a crime"].str.contains("probation, lifetime sex offender registration", na=False),
        df["Punishment for a crime"].str.contains("community service", na=False),
    ],
    [
        2, # probation, lifetime sex offender registration
        1, # community service
    ],
    default=0
)
df

,Date of the Crime,Defendants,Punishment for a crime,Legally Exonerated Encoded,assault,conspiracy,robbery,sexual assault,theft,Location of the Crime_Alabama,...,Location of the Crime_South Carolina,Location of the Crime_Tennessee,Location of the Crime_Texas,Location of the Crime_Virginia,Location of the Crime_Washington,Location of the Crime_West Virginia,Location of the Crime_Wisconsin,death,life sentence,alternative punishment
0,1805,2,death,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,1843,1,death,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,1855,1,death,0,1,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
3,1863,1,death,1,1,0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,0
4,1872,1,death,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,2016,1,20 years,1,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
174,2017,1,12.5 years,1,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,0
175,2017,1,6 years,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
176,2017,1,50 years,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [60]:
df["imprisonment"] = (
    df["Punishment for a crime"]
    .str.extract(r"(\d+(?:\.\d+)?)\s*years", expand=False)
    .pipe(pandas.to_numeric)
)
df.loc[df["imprisonment"].isna(), "imprisonment"] = 0
df

,Date of the Crime,Defendants,Punishment for a crime,Legally Exonerated Encoded,assault,conspiracy,robbery,sexual assault,theft,Location of the Crime_Alabama,...,Location of the Crime_Tennessee,Location of the Crime_Texas,Location of the Crime_Virginia,Location of the Crime_Washington,Location of the Crime_West Virginia,Location of the Crime_Wisconsin,death,life sentence,alternative punishment,imprisonment
0,1805,2,death,1,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0.00
1,1843,1,death,1,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0.00
2,1855,1,death,0,1,0,0,0,0,0,...,0,0,0,1,0,0,1,0,0,0.00
3,1863,1,death,1,1,0,0,0,0,0,...,0,1,0,0,0,0,1,0,0,0.00
4,1872,1,death,1,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,2016,1,20 years,1,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,20.00
174,2017,1,12.5 years,1,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,12.50
175,2017,1,6 years,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,6.00
176,2017,1,50 years,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,50.00


In [61]:
df_without_processing["Punishment for a crime"] = df["Punishment for a crime"]
df = df.drop(columns=["Punishment for a crime"])
df

,Date of the Crime,Defendants,Legally Exonerated Encoded,assault,conspiracy,robbery,sexual assault,theft,Location of the Crime_Alabama,Location of the Crime_Arizona,...,Location of the Crime_Tennessee,Location of the Crime_Texas,Location of the Crime_Virginia,Location of the Crime_Washington,Location of the Crime_West Virginia,Location of the Crime_Wisconsin,death,life sentence,alternative punishment,imprisonment
0,1805,2,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0.00
1,1843,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0.00
2,1855,1,0,1,0,0,0,0,0,0,...,0,0,0,1,0,0,1,0,0,0.00
3,1863,1,1,1,0,0,0,0,0,0,...,0,1,0,0,0,0,1,0,0,0.00
4,1872,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,2016,1,1,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,20.00
174,2017,1,1,0,0,0,1,0,0,0,...,0,0,0,0,0,1,0,0,0,12.50
175,2017,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,6.00
176,2017,1,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,50.00


# Кластеризация

## Масштабирование

In [62]:

scaler = StandardScaler()
X = scaler.fit_transform(df)
X

array([[-5.09225036,  1.45637932,  0.40422604, ..., -0.68316619,
        -0.10107257, -0.57442933],
       [-3.98912515, -0.27185747,  0.40422604, ..., -0.68316619,
        -0.10107257, -0.57442933],
       [-3.64076982, -0.27185747, -2.47386338, ..., -0.68316619,
        -0.10107257, -0.57442933],
       ...,
       [ 1.06202711, -0.27185747,  0.40422604, ..., -0.68316619,
        -0.10107257, -0.24904784],
       [ 1.06202711, -0.27185747,  0.40422604, ..., -0.68316619,
        -0.10107257,  2.13708306],
       [ 1.06202711, -0.27185747,  0.40422604, ..., -0.68316619,
        -0.10107257, -0.53375664]], shape=(178, 50))

## Метод локтя

In [63]:
inertia = []
K = range(1, 101)

for k in K:
    kmeans = KMeans(
        n_clusters=k,
        init="k-means++",
        random_state=42
    )
    kmeans.fit(X)
    inertia.append(kmeans.inertia_)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=list(K),
        y=inertia,
        mode="lines+markers",
        marker=dict(size=8),
        name="Inertia"
    )
)

fig.update_layout(
    title="Метод локтя для K-means++",
    xaxis_title="Число кластеров (k)",
    yaxis_title="Инерция"
)

pio.renderers.default = "vscode"
fig.show()

Из данных графика было принято решение в качестве числа кластеров выбрать значение 41, так как до него инерция на каждом шаге уменьшалась минимум на 100, а после 41 это значение всегда меньше 100

In [64]:
kmeans = KMeans(
    n_clusters=41,
    init="k-means++",
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)
df

,Date of the Crime,Defendants,Legally Exonerated Encoded,assault,conspiracy,robbery,sexual assault,theft,Location of the Crime_Alabama,Location of the Crime_Arizona,...,Location of the Crime_Texas,Location of the Crime_Virginia,Location of the Crime_Washington,Location of the Crime_West Virginia,Location of the Crime_Wisconsin,death,life sentence,alternative punishment,imprisonment,cluster
0,1805,2,1,1,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0.00,17
1,1843,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0.00,21
2,1855,1,0,1,0,0,0,0,0,0,...,0,0,1,0,0,1,0,0,0.00,15
3,1863,1,1,1,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0.00,10
4,1872,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0.00,29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,2016,1,1,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,20.00,25
174,2017,1,1,0,0,0,1,0,0,0,...,0,0,0,0,1,0,0,0,12.50,28
175,2017,1,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,6.00,12
176,2017,1,1,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,50.00,36


# Выводы

## Визуализация

In [65]:
tsne = TSNE(
    n_components=2,
    perplexity=10,
    learning_rate="auto",
    init="pca",
    random_state=42
)
X_tsne = tsne.fit_transform(X)

custom_data = [
    df_without_processing["Defendants"].values,
    df_without_processing["Date of the Crime"].values,
    df_without_processing["Type of the Crime"].values,
    df_without_processing["Punishment for a crime"].values,
    df_without_processing["Location of the Crime"].values,
    df_without_processing["Legally Exonerated Encoded"].values
]

fig = px.scatter(
    df,
    x=X_tsne[:, 0],
    y=X_tsne[:, 1],
    color="cluster",
    color_discrete_sequence=px.colors.qualitative.Bold,
    color_continuous_scale=px.colors.diverging.Portland,
    custom_data=custom_data,
    title="K-means++ (PCA)",
    labels={
        "cluster": "Cluster",
    }
)

fig.update_traces(
    hovertemplate=(
        "<b>X</b>: %{x}<br>"
        "<b>Y</b>: %{y}<br>"
        "<b>Cluster</b>: %{marker.color}<br>"
        "<b>Defendants</b>: %{customdata[0]}<br>"
        "<b>Date of the Crime</b>: %{customdata[1]}<br>"
        "<b>Type of the Crime</b>: %{customdata[2]}<br>"
        "<b>Punishment for a crime</b>: %{customdata[3]}<br>"
        "<b>Location of the Crime</b>: %{customdata[4]}<br>"
        "<b>Legally Exonerated Encoded</b>: %{customdata[5]}<br>"
    )
)

fig.show()

## Размер кластеров

In [66]:
cluster_sizes = df["cluster"].value_counts().sort_index()

fig = px.bar(
    x=cluster_sizes.index,
    y=cluster_sizes.values,
    labels={"x": "Cluster", "y": "Count"},
    title="Размеры кластеров"
)

fig.show()